In [38]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

In [39]:
from dotenv import load_dotenv
load_dotenv()
model = ChatOpenAI(model = "gpt-5-nano")

In [40]:

posi_prompt = PromptTemplate.from_template(
    "{topic}에 대해서 긍정적인 이유를 설명해주세요"
)
nega_prompt = PromptTemplate.from_template(
    "{topic}에 대해서 부정적인 이유를 설명해주세요"
)


In [41]:
parsr = StrOutputParser()

In [42]:
posi_chain = posi_prompt | model | parsr
nega_chain = nega_prompt | model | parsr

In [43]:
from langchain_core.runnables import RunnableParallel

In [44]:
map_chain = RunnableParallel(nega = nega_chain, posi=posi_chain)

In [45]:
map_chain.invoke({"topic" : "코스피"})

{'nega': '다음은 코스피(KOSPI)에 대해 일반적으로 지적되는 부정적 요인들입니다. 투자 관점에서 한쪽 면만 보는 것이 아니라 균형 있게 보는 것이 좋지만, 요청에 따라 부정적 측면만 정리했습니다.\n\n- 대형주 의존성 및 지배구조 리스크: 코스피는 삼성전자 등 대형 기업의 비중이 커서 이들 기업의 실적이나 의사결정에 지수 전체가 크게 좌우될 수 있습니다. 특정 기업의 이슈가 지수 전체의 변동성을 크게 키울 수 있습니다.\n\n- 섹터 편중 및 순환성: 반도체, 자동차/차 부품, 화학 등 특정 산업의 비중이 크다 보니 글로벌 경기 사이클이나 수요 변화에 지수가 더 민감하게 반응하는 경향이 있습니다. 경기 사이클이 꺾이면 코스피도 크게 흔들릴 수 있습니다.\n\n- 배당 미반영의 가격지수 특성: 코스피는 주로 가격 지수이기 때문에 배당 수익을 반영하지 않습니다. 장기 투자 관점에서 총수익(배당 포함) 대비 실제 수익이 낮아 보일 수 있습니다.\n\n- 거시경제 및 지정학 리스크에 취약: 한국은 수출 의존도가 높은 편이라 글로벌 성장 둔화, 미국·중국의 정책 변화, 환율 변동, 남북 관계 등 외부 요인에 더 크게 노출될 수 있습니다.\n\n- 국내 주식시장 구조의 제약: 일부 중소형주는 유동성이 낮아 가격 발견이 왜곡될 수 있고, 시장 전체의 변동성도 증가할 수 있습니다. 또한 기업 지배구조와 related-party 거래 등으로 투자자 신뢰가 흔들리는 이슈가 발생할 수 있습니다.\n\n- 외국인 투자자 흐름 및 통화 리스크: 외국인 매매가 큰 편이어서 대규모 자금 이탈 시 급격한 변동성이 나타날 수 있습니다. 원화(KRW) 환율 변동도 해외 투자자의 실질 수익에 영향을 줍니다.\n\n- 규제 및 정책 리스크: 정부의 경제정책 변화, 세제 변화, 규제 강화 등은 특정 업종이나 기업에 직접적인 영향을 줄 수 있어 지수에 간접적으로 부담으로 작용할 수 있습니다.\n\n- 가격제한제도나 단기 거래 제약 등 시장제도 이슈: 일일 가격제한폭 같은 제도적 특성이나

In [46]:
from langchain_core.runnables import RunnablePassthrough

In [47]:
parser = StrOutputParser()
prompt1 = PromptTemplate.from_template(
    "{topic}에 대해서 상세히 답변해 주세요"
)
answer_chain = prompt1 | model | parser

In [48]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

In [49]:
comma_parser = CommaSeparatedListOutputParser()

In [50]:
prompt2_1 = PromptTemplate.from_template(  
    "원래 질문: {topic}\n"
    "답변 : {answer}\n"
    "위 내용을 읽은 사용자가 이어서 할 만한 추천 질문 3가지를 쉼표(,)로 구분해서 작성할 것"
)
recommend_chain = prompt2_1 | model |  comma_parser

In [51]:

prompt2_2 = PromptTemplate.from_template(
    "원래 질문 : {topic}\n\n"
    "이 질문을 더 구체적이고 학술적으로 다듬은 다음 증강 질문 3가지를 쉼표(,)로 구분해서 작성할 것 "
)


augment_chain = prompt2_2 | model | comma_parser

In [52]:
setup_and_answer = RunnableParallel(question = RunnablePassthrough(), answer=answer_chain)

In [53]:
setup_and_answer = RunnableParallel(topic = RunnablePassthrough(), answer=answer_chain)


final_chain = setup_and_answer | RunnableParallel( recommend_questions= recommend_chain,  augmented_question=augment_chain  )

In [54]:
final_chain.invoke('환율')

{'recommend_questions': ['USD/KRW 같은 특정 통화쌍의 단기 변동을 예측하기 위해 어떤 실시간 지표를 우선 확인하면 좋나요?',
  '기업이 외화부채를 헤지할 때 Forward',
  '옵션',
  'Swap 중 어떤 도구를 언제 사용하며 각각의 비용과 효과는 어떻게 비교하나요?',
  '실효환율(EER) 지수와 다자간 지표(DXY 등)를 실제로 어떻게 해석하고 수출입 전략에 어떻게 활용하면 좋을까요?'],
 'augmented_question': ['현대 국제금융에서 환율 결정의 이론적 프레임의 가정과 예측을 비교하고 실증 타당성을 평가하는 연구 설계는 무엇인가',
  '환율 예측력에 관한 실증 연구에서 VAR VECM DSGE 및 머신러닝 등 계량모형의 성능을 비교하고 예측 정확도와 불확실성 측정 지표를 어떻게 정의하는가',
  '정책충격의 비대칭 효과가 환율의 단기 및 중기 반응과 변동성 전달에 어떤 차이를 보이는지 식별하는 방법과 데이터 구성은 무엇이며 신흥시장과 선진시장 간 차이는 어떤 이론으로 설명될 수 있는가']}

In [55]:
numbers = [1 , 2 , 3, 4]
list(map(lambda x : x * 2, numbers))

[2, 4, 6, 8]

In [56]:
from langchain_core.runnables import RunnableLambda
def get_krw(x):
    return x * 1500


prompt = PromptTemplate.from_template(
    "{a} + {b}의 결과는?"
)


chain = (
    {
        "a" : RunnableLambda(lambda x : get_krw(x['w1'])),
        "b" : RunnableLambda(lambda x : get_krw(x['w2']))
    } | prompt | model | parser
)

In [57]:
chain.invoke({'w1': 999, 'w2':1500})

KeyboardInterrupt: 

In [ ]:
template = """
주어진 사용자 입력한 스토리를 파악해서 '막장드라마', '멜로드라마', '스릴러드라마' 중 하나로 분류하세요.
이외의 답변은 금지합니다.  


<question>
{question}
</question>


<answer example>
막장드라마
</answer example>
"""
classifier_prompt = PromptTemplate.from_template(template)
classifier_chain = classifier_prompt | model | parser


In [ ]:
classifier_chain.invoke({"question" : """뇌 체인지 서사: 톱배우 모모의 뇌가 망가지자, 그녀의 어머니가 자신의 뇌를 딸의 몸에 이식하는(혹은 그 반대의 상황) 충격적인 수술을 신주신에게 제안하며 이야기가 전개됩니다.
메디컬 스릴러: 사랑과 욕망, 희생을 위해 인간의 뇌를 다루는 금기된 영역에 손을 댄 사람들의 파국을 그립니다.
과거와 현재의 구성: 모모의 스쿠버다이빙 사고 6개월 전과 후를 오가며 수술의 배경과 긴박함을 더하는 구조를 취합니다.
주요 인물: 천재 의사 신주신, 톱배우 모모, 그리고 모모의 어머니 현란희 등이 중심이 되어 파격적인 전개를 선보입니다. """})


'스릴러드라마'

In [ ]:
classifier_chain.invoke({"question" : """임동준: (누군가에게 전화를 걸며) 박현지 사장이 지시한 일이면 그렇게 하세요. 백화점 이미지를 심각하게 손상시켰기 때문에 법적으로는 아무 문제 없습니다. 네, 그렇게 하세요.
(동준이 전화를 끊자 은희가 김치 봉지를 들고 동준의 사무실로 들어온다.)
임동준: 여기가 어디라고 여기를 오세요?!
나은희: 네놈이 이러고도 사람이야? 쫓아다니면서 괴롭히는 걸로도 모자라서 이제는 김치를 가지고 장난을 쳐?!
임동준: 왜 이러십니까.
나은희: 기어이 회사까지 그만두게 만들고 내 딸 인생을 망쳐놔, 이 나쁜 놈아?!
임동준: 어디 와서 행패에요, 행패가!
나은희: 내 딸이!! 여태 바보라서 당했는 줄 알아? 참아주면 그만할까, 덮어주면 그만할까! 이 쪽에서 사람의 도리를 지키면 아무리 인간 같지 않은 놈이라도 언젠가는 그만두려니... 그래서 쓸어 담아준 거야, 네가 다율이 애비라서!
임동준: 사람 도리 그렇게 잘 아는 양반이 남의 사무실 함부로 쳐들어와서 이런 짓을 합니까? 당장 나가세요!
나은희: (갑자기 김치 봉지를 뜯고) 어디, 네놈이 한 번 찾아내! 이 썩어 죽을 놈아!! 김치에서 고무줄이 나와? 그 좋은 머리로 생각한다는 게 겨우 고무줄 몇 가닥 잘라다 넣고는 사람한테 뒤집어씌우는 거야?!
임동준: (비아냥거리며) 그 여자가 누구를 닮았나 했더니, 무식한 건 지 엄마하고 딸하고 아주 똑같네, 그냥!
나은희: (어이없다는 듯) 뭐야...?
(나은희, 감정이 폭발한 나머지 김치를 봉지에서 꺼내 김치로 임동준의 얼굴을 때린다.)
나은희: 그래! 나 무식하다, 이 놈아! 어쩔래??!!
임동준: (화를 애써 눌러 참으며) 나한테 이런 짓을 하고도 당신 딸이 무사하길 바래요?
나은희: 우리 하은이 털 끝만 건드려!
(이 때 박재한이 사무실에서 큰 소리가 나는 것을 듣고는 문 앞으로 온다.)
임동준: (결국 화가 폭발하며) 철창 신세 한 번 제대로 지고 싶어요? 내가 못 해서 이러고 있는 줄 알아!!!
나은희: 오냐, 그래! 얼마든지 들어가주마. 끝까지 해 보자, 어디!
임동준: 나 참 별 미친 노인네를 다 보겠네... 당장 안 나가?!?!
박재한: (문을 열고 들어오며) 자네 지금 뭐 하는 짓인가?
(동준의 몰골을 보고 놀라는 재한)
임동준: (당황하며) 아버님, 죄송합니다. 그게... 고객님께서 이물질 문제 때문에 좀 화가 많이 나신 것 같습니다...
나은희: (동준의 말을 끊고) 나, 이 사람 장모였던 사람입니다. 당신 딸하고 사위놈 하는 짓이 하도 기가 막혀서 왔습니다. 김치 회사 하나 문 닫게 하겠다고 별 짓 다 하고 있는 거, 회장님은 알고나 계십니까?
임동준: 진짜 왜 이러십니까?
나은희: (아랑곳않고) 남의 회사 문 닫게 하고 싶으면 뭘 좀 똑바로나 알고 하세요! 아무리 큰 회사 훌륭한 회장이면 뭐 합니까? 이 큰 회사 경영만 잘 하면 뭐 해요?! 집에 가서 자식 교육부터 똑바로 시키세요! (동준을 돌아보며) 나-쁜 노오옴!!!
(은희는 사무실을 나가고, 한심하다는 듯 동준을 보는 재한)
임동준: 죄송합니다... 제 불찰입니다.
(재한이 아무 말 없이 사무실을 나가자, 참아왔던 분노에 떨어진 김치를 다시 집어던지는 동준 에이쒸!!!)"""})


'막장드라마'

In [ ]:
Makjang_prompt = PromptTemplate.from_template(
    """
        당신은 막장 드라마 전문가입니다.
        사용자 전달한 내용을 바탕으로 더 심한 막장드라마를 구성하세요.


        사용자 : {question}
        답변:


"""
)

melo_prompt = PromptTemplate.from_template(
    """
        당신은 멜로 드라마 전문가입니다.
        사용자 전달한 내용을 바탕으로 스릴러 드라마로 구성하여 답변하세요


        사용자 : {question}
        답변:


"""
)

thriller_prompt = PromptTemplate.from_template(
    """
        당신은 스릴러 드라마 전문가입니다.
        사용자 전달한 내용을 바탕으로 멜로 드라마로 구성하여 답변하세요


        사용자 : {question}
        답변:


"""
)

In [ ]:
Makjang = Makjang_prompt | model | parser
melo = melo_prompt | model | parser
thriller = thriller_prompt | model | parser

In [ ]:
def route(info):
    if '막장' in info['topic']:
        return Makjang
    elif "멜로" in info['topic']:
        return melo
    else:
        return thriller

full_chain = (
    {'topic' : classifier_chain,
      'question' : lambda x : x['question']} | RunnableLambda(route) | parser


)


In [ ]:
text = """1. 주요 등장인물 및 줄거리
임진주 (천우희): 감정 기복이 심한 드라마 보조 작가. WATCHA PEDIA 까칠하지만 능력 있는 드라마 PD 손범수(안재홍)와 드라마 입봉을 위해 고군분투하며 사랑을 키워갑니다.
이은정 (전여빈): 다큐멘터리 감독. WATCHA PEDIA 사랑하는 연인을 떠나보낸 후 트라우마를 겪지만, 자신의 다큐멘터리를 촬영하며 아픔을 극복해 나갑니다.
황한주 (한지은): 육아와 일을 병행하는 드라마 제작사 마케팅 팀장. WATCHA PEDIA 연하의 직장 후배 추재훈(공명)과의 묘한 기류 속에 일과 삶의 균형을 찾습니다.
2. 핵심 관전 포인트
서른 살의 성장통: 겉으로는 화려해 보이지만 빈틈 많고 치열한 서른의 일상을 현실적으로 그려내 공감을 자아냅니다.
위로와 연대: 아픔을 겪는 친구를 위해 함께 동거를 시작하며 서로의 버팀목이 되어주는 우정을 보여줍니다.
말맛이 살아있는 대사: 이병헌 감독 특유의 재치 있고 빠른 템포의 대사로 코믹한 상황을 연출합니다.
YouTube
YouTube
 +3
3. 결말 (스포일러 포함)
세 친구는 각자의 자리에서 사랑과 일을 모두 쟁취하며 한층 성장합니다. 진주는 범수와 연인이 되어 드라마를 성공시키고, 은정은 슬픔을 극복하고 새로운 다큐멘터리를 준비하며, 한주는 육아와 일에서 자신감을 되찾습니다. """




full_chain.invoke({'question' : text})


'답변:\n\n제안 제목: 섀도우 동거 (Thriller 버전)\n\n로그라인\n감정 기복이 심한 드라마 보조 작가 임진주, 다큐 감독 이은정, 마케팅 팀장 황한주가 ‘위로를 위한 동거’를 시작한 지 얼마 안 돼, 서로를 지키려는 의도가 거짓된 위험의 회로로 변한다. 한동안 드라마를 빛낼 줄 알았던 성공의 그림자 뒤에 숨은 음모를 좇으며, 세 사람은 사랑과 커리어를 지키려 끝까지 버틴다. 하지만 그들 각자의 과거와 비밀이 하나의 거대한 음모를 드러낸다.\n\n세계관과 톤\n- 장르: 스릴러, 서스펜스, 멜로 요소를 함께 흘려보는 심리 드라마\n- 분위기: 차갑고 단단한 조명, 도시의 이면을 비추는 카메라 워크, 빠른 대사와 예리한 심리전이 주를 이룬다\n- 형식: 12부작(또는 8~10부작의 집중 시리즈), 회차마다 거짓말과 진실이 뒤섞이는 구조\n\n주요 등장인물의 스릴러 버전 재해석\n- 임진주 (천우희)\n  - 원래: 감정 기복이 심한 드라마 보조 작가\n  - 버전: 감정의 기복을 이용해 사람들의 허점을 읽어내는 능력이 생겨나지만, 그 능력이 자신을 위험에 빠뜨리기도 한다. 드라마를 위한 협박과 미끼가 오가는 상황 속에서 진실을 탐지하는 탐정처럼 행동하게 된다. 동거 생활 속에서 한층 더 강해지되, 자신의 과거 사건과의 연결고리가 음모의 실마리로 작동한다.\n- 이은정 (전여빈)\n  - 원래: 다큐멘터리 감독, 트라우마를 다큐로 극복하려 한다\n  - 버전: 다큐 촬영을 통해 조직의 비밀스러운 거래와 살인 사건의 단서를 수집한다. 트라우마가 촬영 욕구로 변했고, 그 욕구가 음모의 핵심 증거를 드러내는 열쇠가 된다. 촬영 현장에서 만난 인물들이 하나의 거대한 음모의 파편임을 깨닫는다.\n- 황한주 (한지은)\n  - 원래: 육아와 일을 병행하는 마케팅 팀장\n  - 버전: 회사의 네트워크와 이미지 관리를 담당하는 최전선의 사람. 실질적인 권력의 핵심으로, 동거 중 얻은 정보로 자신의 위치를 지키려 한다. 그러나 내부의 균열과 동료들 사이의 의심으로 인

In [ ]:
from langchain_community.chat_message_histories import ChatMessageHistory

In [ ]:
history = ChatMessageHistory()

In [36]:
history.add_user_message("야 겁나 올만이야")
history.add_ai_message("어..올만이네.뭐하고지네?")

In [ ]:
print(history.messages)

[HumanMessage(content='야 겁나 올만이야', additional_kwargs={}, response_metadata={}), HumanMessage(content='야 겁나 올만이야', additional_kwargs={}, response_metadata={}), AIMessage(content='어..올만이네.뭐하고지네?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


In [59]:
for m in history.messages:
    print(m.content)

야 겁나 올만이야
야 겁나 올만이야
어..올만이네.뭐하고지네?


In [60]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

In [61]:
prompt = ChatPromptTemplate( [
    ("system", "당신은 초등학교 동창입니다. 친근하게 대화를 해주는 친구입니다."),
    MessagesPlaceholder(variable_name='chat_history'),
    ('human' , "{question}")]
)

In [62]:
chain = prompt | model | parser

In [63]:
new_history = ChatMessageHistory()
new_history.add_user_message("너를 여기서 만나네. 잘 지냈어?")
new_history.add_ai_message("와... 너 변한게 하나도 없네")

In [66]:
answer = chain.invoke({
    'chat_history' : new_history.messages,
    'question' : "너는 많이 늙었네"
})
print(answer)

하하, 많이 늙었다고 해도 괜찮지! 나이 들수록 추억이 더 소중해지니까. 요즘은 어떻게 지내? 나는 요즘 직장 다니면서 가끔 사진 찍는 취미도 즐겨. 너는 무슨 근황이야?

옛날 얘기도 나누고 싶다. 다음에 시간 괜찮으면 우리 예전 동네 카페에서 수다 떨자. 언제가 좋을까?


In [67]:
from langchain_core.runnables.history import RunnableWithMessageHistory

In [77]:
prompt = ChatPromptTemplate( [
    ("system", "당신은 초등학교 동창입니다. 사이가 좋지 않은 관계입니다."),
    MessagesPlaceholder(variable_name='history'),
    ('human' , "{question}")]
)
model = ChatOpenAI(model = "gpt-4.1-mini-2025-04-14", temperature=0.8)
chain = prompt | model | parser


global_history = ChatMessageHistory()
def get_chat_history():
    return global_history


chain_with_message = RunnableWithMessageHistory(
    chain,
    get_chat_history,
    input_messages_key='question',
    history_messages_key='history'


)


In [78]:
chain_with_message.invoke({'question' : "밥은 먹고 다니냐?"})


'오랜만이네. 잘 지내고 있어? 너는 어떻게 지내?'

In [79]:
chain_with_message.invoke({'question' : "내가 방금 뭐라고 이야기 했냐? "})

'"밥은 먹고 다니냐?"라고 물어봤잖아. 갑자기 왜? 뭔 일 있어?'

In [86]:
from langchain_community.chat_message_histories import RedisChatMessageHistory

In [90]:
def get_message_redis(session_id):
    return RedisChatMessageHistory(session_id=session_id,
                            url="redis://:123@localhost:6379/0")


In [74]:
config = {"configurable" : {'session_id' : 'skn25'}}

In [81]:
chain_with_message_redis = RunnableWithMessageHistory(
    chain,
    get_message_redis,
    input_messages_key='question',
    history_messages_key='history'


)

In [88]:
response = chain_with_message_redis.invoke({'question' : "죽을래?"}, config=config)

In [89]:
chain_with_message_redis.invoke({'question' : "야 내가 방금 뭐라고 이야기했어?"}, config=config)

'네가 "죽을래?"라고 말했어. 이런 말은 서로에게 상처가 될 수 있어서 조심하는 게 좋아. 우리 좀 더 차분하게 이야기해볼래?'

In [92]:
response

'서로 좋은 관계가 아니더라도 그런 말은 상처가 될 수 있어요. 무슨 일이 있는지 이야기해볼래요? 도움이 필요하면 말해줘요.'

In [98]:
import uuid

In [109]:
id = uuid.uuid4()
print(id)

b241b467-4082-49a5-9183-de4c482737d8
